# 02 CBAM 通道注意力详解

上一讲认识了 CBAM 的两个阶段。这一讲只研究**通道注意力**：它的输入是什么，通道权重如何算出来，权重如何影响特征，以及 MLP 是怎样学习的。

## 1. 先分清图像通道和特征通道

RGB 图片的 3 个通道表示红、绿、蓝。图片经过 CNN 卷积后，会得到一组特征图，通道数由网络结构决定，例如 64 个输出通道就得到 64 张特征图。这些通道不是 64 种颜色，也不一定各自对应明确的语义。

CBAM 通道注意力通常接收的是 **CNN 的中间特征图**，而不是直接处理 RGB 图片。不同通道对当前任务的帮助可能不同，通道注意力尝试根据当前输入调整各个通道。

## 2. 输入、权重和输出

设输入特征为 $F$，其中 $B$ 是样本数，$C$ 是通道数，$H$、$W$ 是高和宽：

$$F \in \mathbb{R}^{B \times C \times H \times W}$$

通道注意力为**每个样本的每个通道**生成一个权重：

$$M_c(F) \in \mathbb{R}^{B \times C \times 1 \times 1}$$

权重乘回输入后，输出形状仍是 $B \times C \times H \times W$。例如输入为 $2 \times 64 \times 28 \times 28$，权重是 $2 \times 64 \times 1 \times 1$，输出仍是 $2 \times 64 \times 28 \times 28$。两张图片各自有一组 64 个权重。

## 3. 第一步：用池化概括每个通道

每个通道有 $H \times W$ 个数。为了给整个通道生成一个权重，先分别做**空间维度上的全局平均池化和全局最大池化**。

例如某个通道为：

$$\begin{bmatrix}1 & 2\\3 & 6\end{bmatrix}$$

平均池化得到 $(1+2+3+6)/4=3$；最大池化得到 $6$。前者概括整体响应，后者保留最强响应。每个通道都这样操作，得到两组通道描述：

$$F_{\mathrm{avg}}^c, F_{\mathrm{max}}^c \in \mathbb{R}^{B \times C \times 1 \times 1}$$

例如 64 个通道分别得到 64 个平均值和 64 个最大值。**这里压缩的是高和宽，保留的是通道维度。**

## 4. 第二步：两组描述分别通过共享 MLP

两组池化结果分别输入**同一个 MLP**，也就是两条分支使用同一组可训练参数。MLP 学习结合通道信息，产生每个通道的分数。

原始 CBAM 中，MLP 通常先把通道维度从 $C$ 缩小到 $C/r$，经过 ReLU，再恢复到 $C$；$r$ 是预先设置的缩减比例。例如 $C=64$、$r=16$ 时：

$$64 \rightarrow 4 \rightarrow 64$$

这是每个样本的一组通道描述的维度变化；MLP 最终仍为每个通道给出一个分数。缩减维度有助于减少参数量。实际实现需要保证中间维度是合法的正整数。

## 5. 第三步：分数相加，再经过 Sigmoid

假设只有 3 个通道，两条分支经过共享 MLP 后的结果**举例为**：

$$[1.0,\ 0.2,\ 0.4] + [0.5,\ 0.1,\ 0.6] = [1.5,\ 0.3,\ 1.0]$$

对应分数相加后，对每个分数应用 Sigmoid：

$$\sigma([1.5,\ 0.3,\ 1.0]) \approx [0.82,\ 0.57,\ 0.73]$$

这三个数就是本次输入的通道权重。完整过程可写成：

$$M_c(F)=\sigma\left(\operatorname{MLP}(\operatorname{AvgPool}(F))+\operatorname{MLP}(\operatorname{MaxPool}(F))\right)$$

池化值较大**不保证**最后权重一定较大，因为 MLP 还有可学习参数，并会结合各通道的信息。上面的 MLP 输出是为了说明相加和 Sigmoid 而假设的，并非从某组给定 MLP 参数算出的。

## 6. 第四步：广播并乘回原特征

每个权重应用于对应通道的所有空间位置。例如某通道为：

$$\begin{bmatrix}1 & 2\\3 & 4\end{bmatrix},\quad \text{权重}=0.8$$

逐元素相乘后得到：

$$\begin{bmatrix}0.8 & 1.6\\2.4 & 3.2\end{bmatrix}$$

整体记作：

$$F'=M_c(F)\odot F$$

$\odot$ 表示逐元素相乘；形状中的 $1 \times 1$ 通过广播应用到对应通道的 $H \times W$ 个位置。权重越大，通道受到的抑制越少；最终预测受到多大影响还取决于原始特征和后续网络。Sigmoid 权重处于 0 和 1 之间，所以“突出”通常指相对其他被压低更多的特征。

## 7. MLP 的参数和通道权重不是一回事

MLP 的**参数**在模型创建时初始化，并在训练中更新；通道**权重**则是在每次前向传播时，使用当前输入的池化结果和当前 MLP 参数计算出来的。参数不变时，换一张图片，得到的通道权重也可能不同。

训练时，调整后的特征继续经过后续网络得到预测；预测和标签计算损失。反向传播沿计算图求出 MLP 参数对损失的梯度，再由优化器更新参数。平均池化和最大池化两条分支共享同一组 MLP 参数，来自两条分支的梯度会累加到这组参数上。

因此 MLP 不需要单独训练，它和 CNN 等其他可训练部分一起接受任务损失的监督。训练目标是改善任务表现，并不保证每个通道权重都能独立解释最终预测。

## 8. 与空间注意力区分

| 对比 | 通道注意力 | 空间注意力 |
|---|---|---|
| 回答的问题 | 哪些通道相对更值得保留？ | 哪些空间位置相对更值得保留？ |
| 池化方向 | 沿高和宽 | 沿通道 |
| 每个样本的权重形状 | $C \times 1 \times 1$ | $1 \times H \times W$ |
| 权重如何应用 | 每个通道一个权重 | 每个位置一个权重，所有通道共享 |

在原始 CBAM 中，通道注意力的输出接着进入空间注意力。

## 9. 小结与自测

通道注意力先从每个通道提取平均和最大响应，再用共享 MLP、相加和 Sigmoid 生成权重，最后把权重乘回原特征。输入、输出形状相同，但数值通常改变。

1. 输入为 $B \times C \times H \times W$，池化后、权重和输出各是什么形状？
2. 平均池化和最大池化分别沿哪些维度操作？
3. 共享 MLP 的参数与通道权重有什么区别？
4. 为什么特征图的值更大，不一定意味着通道权重更大？
5. 权重为 0.9 的通道一定比权重为 0.2 的通道对最终预测影响更大吗？为什么？
6. 两条池化分支的 MLP 参数是如何得到更新的？